In [1]:
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv('spam.csv',encoding="latin-1")
df.head()

,Message,Class
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [5]:
df['length']=df['Message'].apply(len)
df.head()

,Message,Class,length
0,"Go until jurong point, crazy.. Available only ...",ham,111
1,Ok lar... Joking wif u oni...,ham,29
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam,155
3,U dun say so early hor... U c already then say...,ham,49
4,"Nah I don't think he goes to usf, he lives aro...",ham,61


## Preprocessing

In [20]:
import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import string

def preprocess(spam):
    msg=spam.translate(str.maketrans('','',string.punctuation))
    msg=[w for w in msg.split() if w.lower() not in stopwords.words("english")]

    spam_words=""
    
    for m in msg:
        stemmer=SnowballStemmer("english")
        spam_words+=(stemmer.stem(msg))+" "
    return spam_words

In [31]:
# features=df["Message"].copy()
# features=features.apply(preprocess)
# features
tf=pd.read_csv('Text features.csv')
tf.head()

,Unnamed: 0,Message
0,0,go jurong point crazi avail bugi n great world...
1,1,ok lar joke wif u oni
2,2,free entri 2 wkli comp win fa cup final tkts 2...
3,3,u dun say earli hor u c alreadi say
4,4,nah dont think goe usf live around though


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer()
features=vectorizer.fit_transform(tf['Message'].fillna(''))
print(features)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 46982 stored elements and shape (5572, 8037)>
  Coords	Values
  (0, 3289)	0.14081218308843002
  (0, 4063)	0.3501112803835596
  (0, 5539)	0.23906338437276378
  (0, 2223)	0.2711370860940892
  (0, 1327)	0.2662313123328933
  (0, 1726)	0.29577542446330046
  (0, 3380)	0.19449555011263384
  (0, 7776)	0.2379272168522122
  (0, 4204)	0.29577542446330046
  (0, 1724)	0.33421906130459
  (0, 2006)	0.29577542446330046
  (0, 3342)	0.164475816719243
  (0, 1138)	0.3501112803835596
  (0, 7574)	0.19449555011263384
  (1, 5170)	0.2827396376113674
  (1, 4239)	0.42078899608869724
  (1, 4029)	0.47731294876998304
  (1, 7688)	0.44480400570972006
  (1, 5200)	0.5633086751818669
  (2, 3108)	0.1156013509460074
  (2, 2769)	0.35890157185696997
  (2, 7734)	0.1899683039634032
  (2, 2101)	0.19609614678919884
  (2, 7701)	0.145407091351942
  (2, 2886)	0.46833156720022556
  :	:
  (5568, 3289)	0.29668586321213597
  (5568, 3634)	0.3752780670596876
  (5568, 3097)	0.

In [42]:
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
df['Class']=label_encoder.fit_transform(df['Class'])
df.head()

,Message,Class,length
0,"Go until jurong point, crazy.. Available only ...",0,111
1,Ok lar... Joking wif u oni...,0,29
2,Free entry in 2 a wkly comp to win FA Cup fina...,1,155
3,U dun say so early hor... U c already then say...,0,49
4,"Nah I don't think he goes to usf, he lives aro...",0,61


In [48]:
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(features,df['Class'],test_size=.2)

In [52]:
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier

dt=DecisionTreeRegressor()
dt.fit(xtrain,ytrain)

dt_pred=dt.predict(xtest)
dt_pred

dtc=DecisionTreeClassifier()
dtc.fit(xtrain,ytrain)
dtc_pred=dtc.predict(xtest)
dtc_pred

array([0, 0, 0, ..., 0, 0, 0])

In [53]:
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier

rfc=RandomForestClassifier()
rfc.fit(xtrain,ytrain)

rfc_pred=rfc.predict(xtest)
rfc_pred

rfr=RandomForestRegressor()
rfr.fit(xtrain,ytrain)

rfr_pred=rfr.predict(xtest)
rfr_pred

array([0.  , 0.27, 0.  , ..., 0.  , 0.  , 0.  ])

In [54]:
from sklearn.svm import SVC,SVR

svc=SVC()
svc.fit(xtrain,ytrain)
svc_pred=svc.predict(xtest)

svr=SVR()
svr.fit(xtrain,ytrain)
svr_pred=svr.predict(xtest)
svr_pred

array([0.07165055, 0.57757731, 0.11517895, ..., 0.08765167, 0.06242402,
       0.03335838])

In [67]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

In [64]:
dt_as,rfr_as,svr_as=accuracy_score(ytest,dt_pred)*100,accuracy_score(ytest,rfr_pred.round())*100,accuracy_score(ytest,svr_pred.round())*100
dt_p,rfr_p,svr_p=precision_score(ytest,dt_pred)*100,precision_score(ytest,rfr_pred.round())*100,precision_score(ytest,svr_pred.round())*100
dt_f1,rfr_f1,svr_f1=f1_score(ytest,dt_pred)*100,f1_score(ytest,rfr_pred.round())*100,f1_score(ytest,svr_pred.round())*100
dt_rs,rfr_rs,svr_rs=recall_score(ytest,dt_pred)*100,recall_score(ytest,rfr_pred.round())*100,recall_score(ytest,svr_pred.round())*100

In [65]:
print('\tDecision Tree Regressor'+'\t'+'RandomForest Regressor'+'\t'+'SVR')
print('accuracy : ',dt_as,'\t',rfr_as,'\t',svr_as)
print('precison : ',dt_p,'\t',rfr_p,'\t',svr_p)
print('f1 score : ',dt_f1,'\t',rfr_f1,'\t',svr_f1)
print('recall score : ',dt_rs,'\t',rfr_rs,'\t',svr_rs)

	Decision Tree Regressor	RandomForest Regressor	SVR
accuracy :  95.15695067264573 	 95.87443946188341 	 97.21973094170404
precison :  85.16129032258064 	 88.74172185430463 	 100.0
f1 score :  83.01886792452831 	 85.35031847133759 	 89.49152542372882
recall score :  80.98159509202453 	 82.20858895705521 	 80.98159509202453


In [73]:
pred=[dt_pred,rfr_pred,svr_pred]
n={0:'Decision Tree',1:'Random Forest',2:'SV'}
i=0
for p in pred:
    print("Classification Report : ",n[i])
    i+=1
    print('-'*100)
    print(classification_report(ytest,p.round()))

Classification Report :  Decision Tree
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       952
           1       0.85      0.81      0.83       163

    accuracy                           0.95      1115
   macro avg       0.91      0.89      0.90      1115
weighted avg       0.95      0.95      0.95      1115

Classification Report :  Random Forest
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       952
           1       0.89      0.82      0.85       163

    accuracy                           0.96      1115
   macro avg       0.93      0.90      0.91      1115
weighted avg       0.96      0.96      0.96      1115

Classification Report :  SV
--------------------------------------

In [74]:
dtc_as,rfc_as,svc_as=accuracy_score(ytest,dt_pred)*100,accuracy_score(ytest,rfc_pred.round())*100,accuracy_score(ytest,svc_pred.round())*100
dtc_p,rfc_p,svc_p=precision_score(ytest,dt_pred)*100,precision_score(ytest,rfc_pred.round())*100,precision_score(ytest,svc_pred.round())*100
dtc_f1,rfc_f1,svc_f1=f1_score(ytest,dt_pred)*100,f1_score(ytest,rfc_pred.round())*100,f1_score(ytest,svc_pred.round())*100
dtc_rs,rfc_rs,svc_rs=recall_score(ytest,dt_pred)*100,recall_score(ytest,rfc_pred.round())*100,recall_score(ytest,svc_pred.round())*100

In [82]:
print(end="\t")
for i in n:
    print(n[i],end="\t") 
print('')
print('accuracy',end="\t")
for p in pred:
    print(accuracy_score(ytest,p.round())*100,end="\t")
print('\nprecision',end="\t")
for p in pred:
    print(precision_score(ytest,p.round())*100,end="\t")
print('\nf1',end="\t")
for p in pred:
    print(f1_score(ytest,p.round())*100,end="\t")
print('\nrecall',end="\t")
for p in pred:
    print(recall_score(ytest,p.round())*100,end="\t")

	Decision Tree	Random Forest	SV	
accuracy	95.15695067264573	95.87443946188341	97.21973094170404	
precision	85.16129032258064	88.74172185430463	100.0	
f1	83.01886792452831	85.35031847133759	89.49152542372882	
recall	80.98159509202453	82.20858895705521	80.98159509202453	

In [83]:
pred_c=[dtc_pred,rfc_pred,svc_pred]
n={0:'Decision Tree',1:'Random Forest',2:'SV'}
i=0
for p in pred_c:
    print("Classification Report : ",n[i])
    i+=1
    print('-'*100)
    print(classification_report(ytest,p.round()))

Classification Report :  Decision Tree
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       952
           1       0.87      0.83      0.85       163

    accuracy                           0.96      1115
   macro avg       0.92      0.90      0.91      1115
weighted avg       0.96      0.96      0.96      1115

Classification Report :  Random Forest
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       952
           1       1.00      0.81      0.89       163

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Classification Report :  SV
--------------------------------------